# Phase 1 — CLIP Baseline

**Goal**: Load a single MVTec test image, score it against text prompts using CLIP, and compute an anomaly score.

This is the foundation everything else builds on.

**Before running**: Make sure your runtime is set to T4 GPU
→ `Runtime → Change runtime type → T4 GPU`

## Step 1 — Install dependencies

In [ ]:
!pip install open-clip-torch timm scikit-learn tqdm -q

## Step 2 — Mount Google Drive (where your MVTec data lives)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Update this path to wherever you put the MVTec data in your Drive
MVTEC_ROOT = '/content/drive/MyDrive/mvtec_anomaly_detection'
CATEGORY = 'bottle'  # start with just bottle

## Step 3 — Imports

In [ ]:
import os
import torch
import open_clip
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
from tqdm import tqdm
from sklearn.metrics import roc_auc_score

# Check we have a GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
# Should print: Using device: cuda

## Step 4 — Load CLIP

In [ ]:
# Load CLIP ViT-B/16 — the same model WinCLIP uses
model, _, preprocess = open_clip.create_model_and_transforms(
    'ViT-B-16',
    pretrained='openai'
)
model = model.to(device)
model.eval()  # inference mode — no gradient updates

tokenizer = open_clip.get_tokenizer('ViT-B-16')

print('CLIP loaded successfully')

## Step 5 — Define your text prompts

These are the descriptions you write once. They describe what normal and anomalous look like.
You'll experiment with different prompt wordings in Phase 3.

In [ ]:
# Normal prompts — what a good bottle looks like
normal_prompts = [
    'a photo of a normal bottle',
    'a photo of a flawless bottle with no defects',
    'a photo of a perfect undamaged bottle',
]

# Anomaly prompts — what a defective bottle looks like
anomaly_prompts = [
    'a photo of a broken bottle',
    'a photo of a damaged bottle with cracks',
    'a photo of a defective bottle with contamination',
]

# Encode all prompts into vectors
with torch.no_grad():
    normal_tokens = tokenizer(normal_prompts).to(device)
    anomaly_tokens = tokenizer(anomaly_prompts).to(device)

    normal_text_features = model.encode_text(normal_tokens)
    anomaly_text_features = model.encode_text(anomaly_tokens)

    # Average across all prompts and normalise to unit length
    normal_text_features = normal_text_features.mean(dim=0)
    normal_text_features /= normal_text_features.norm()

    anomaly_text_features = anomaly_text_features.mean(dim=0)
    anomaly_text_features /= anomaly_text_features.norm()

print(f'Normal text embedding shape: {normal_text_features.shape}')
print(f'Anomaly text embedding shape: {anomaly_text_features.shape}')
# Should print: torch.Size([512]) for both

## Step 6 — Score a single test image

This is the core of the system. We compute how similar the image is to
normal vs. anomaly text embeddings. The difference is the anomaly score.

In [ ]:
def score_image(image_path):
    """
    Given a path to an image, returns an anomaly score.
    Higher score = more likely anomalous.
    """
    # Load and preprocess image
    image = Image.open(image_path).convert('RGB')
    image_tensor = preprocess(image).unsqueeze(0).to(device)  # add batch dimension

    with torch.no_grad():
        # Encode image into the same embedding space as text
        image_features = model.encode_image(image_tensor)
        image_features /= image_features.norm(dim=-1, keepdim=True)

        # Cosine similarity with normal and anomaly text
        sim_normal = (image_features @ normal_text_features).item()
        sim_anomaly = (image_features @ anomaly_text_features).item()

    # Anomaly score = how much more similar to anomaly than normal
    anomaly_score = sim_anomaly - sim_normal
    return anomaly_score, sim_normal, sim_anomaly


# Test on one normal image
test_normal_path = os.path.join(MVTEC_ROOT, CATEGORY, 'test', 'good')
sample_image = list(Path(test_normal_path).glob('*.png'))[0]

score, sim_n, sim_a = score_image(sample_image)
print(f'Image: {sample_image.name}')
print(f'Similarity to normal prompts:  {sim_n:.4f}')
print(f'Similarity to anomaly prompts: {sim_a:.4f}')
print(f'Anomaly score: {score:.4f}  ({"ANOMALY" if score > 0 else "NORMAL"})')

## Step 7 — Evaluate on all test images and compute AUROC

Now we run this on every image in the test set and compute the AUROC score.
This is the number you'll report in your paper.

In [ ]:
def evaluate_category(category_path):
    """
    Runs scoring on all test images for a category.
    Returns anomaly scores and ground truth labels.
    """
    test_path = Path(category_path) / 'test'
    scores = []
    labels = []  # 0 = normal, 1 = anomalous

    for split in sorted(test_path.iterdir()):
        is_anomaly = split.name != 'good'
        for img_path in sorted(split.glob('*.png')):
            score, _, _ = score_image(img_path)
            scores.append(score)
            labels.append(1 if is_anomaly else 0)

    return np.array(scores), np.array(labels)


# Run evaluation
category_path = os.path.join(MVTEC_ROOT, CATEGORY)
print(f'Evaluating {CATEGORY}...')

scores, labels = evaluate_category(category_path)
auroc = roc_auc_score(labels, scores)

print(f'\n--- Results for {CATEGORY} ---')
print(f'Total images: {len(scores)}')
print(f'Normal: {(labels==0).sum()} | Anomalous: {(labels==1).sum()}')
print(f'Image-level AUROC: {auroc:.4f}')
print(f'\nWinCLIP reported ~0.852 for bottle — how does yours compare?')

## Step 8 — Visualise some predictions

In [ ]:
def show_predictions(category_path, n=6):
    """
    Shows a grid of test images with their anomaly scores.
    Green border = correctly identified, Red = wrong.
    """
    test_path = Path(category_path) / 'test'
    samples = []

    for split in sorted(test_path.iterdir()):
        is_anomaly = split.name != 'good'
        images = sorted(split.glob('*.png'))[:2]  # 2 from each type
        for img_path in images:
            score, _, _ = score_image(img_path)
            samples.append((img_path, score, is_anomaly))
        if len(samples) >= n:
            break

    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    axes = axes.flatten()

    for i, (img_path, score, is_anomaly) in enumerate(samples[:n]):
        img = Image.open(img_path).convert('RGB')
        axes[i].imshow(img)
        predicted_anomaly = score > 0
        correct = predicted_anomaly == is_anomaly
        color = 'green' if correct else 'red'
        label = f'Score: {score:.3f}\nTrue: {"Anomaly" if is_anomaly else "Normal"}\nPred: {"Anomaly" if predicted_anomaly else "Normal"}'
        axes[i].set_title(label, color=color, fontsize=9)
        axes[i].axis('off')
        for spine in axes[i].spines.values():
            spine.set_edgecolor(color)
            spine.set_linewidth(3)

    plt.suptitle(f'CLIP Zero-Shot Predictions — {CATEGORY}', fontsize=13)
    plt.tight_layout()
    plt.show()


show_predictions(category_path)

## Summary

You now have a working zero-shot anomaly detector.

**What you built:**
- Loaded CLIP and encoded text prompts into embedding vectors
- Scored images by comparing them to normal vs. anomaly descriptions
- Computed AUROC on the full test set
- Visualised predictions

**What's missing (Phase 2):**
- This only gives an image-level score — it can't say *where* the defect is
- DINOv2 patch features will let us build a pixel-level heatmap

**Next:** Open `02_dinov2_fusion.ipynb`